In [2]:
!pip install contractions num2words

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.5 MB/s eta 0:00:00
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13706 sha256=06fc0b3e04cc175a0676002e246b2e18b06d860f76df04266b41e3630b1c9b1d
  Stored in directory: /root/.cache/pip/wheels/1a/bf/a1/4cee4f7678c68c5875ca89eaccf460593539805c3906722228
Successfully built docopt


In [7]:
# Class 1: Sports & Athletics (Context: Winning/Medals)
doc1 = "The gold medal price is high effort"
doc2 = "Winning a gold medal needs a high jump"
doc3 = "Market for a gold medal is a trade of sweat"
doc4 = "The athlete will trade all for a gold medal"

# Class 2: Finance & Economy (Context: Market/Investment)
doc5 = "The gold bars price is high today"
doc6 = "Investing in gold bars needs a high rate"
doc7 = "Market for gold bars is a trade of money"
doc8 = "The bank will trade all for gold bars"

import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, precision_score
import contractions
from num2words import num2words
import spacy

nlp = spacy.load("en_core_web_sm")

# Your Task: Fill these functions

def preprocess_text(text):
    """
    Make sure to lowercase and remove punctuation.
    """
    text = text.lower()
    text = contractions.fix(text)
    for p in text:
        if p.isdigit():
            text = text.replace(p, num2words(p))
    text_sp = nlp(text)
    text_tokens = [t for t in text_sp if not t.is_punct and not t.is_stop]
    return text_tokens

def vectorize(docs, n_gram_size=1):
    all_ngrams = []
    processed_docs_ngrams = []

    for doc_text in docs:
        processed_tokens = preprocess_text(doc_text)
        doc_ngrams = []
        for i in range(len(processed_tokens) - n_gram_size + 1):
            ngram = " ".join([token.text for token in processed_tokens[i:i+n_gram_size]])
            doc_ngrams.append(ngram)
            all_ngrams.append(ngram)
        processed_docs_ngrams.append(doc_ngrams)

    # Build vocabulary
    vocabulary = sorted(list(set(all_ngrams)))
    ngram_to_idx = {ngram: i for i, ngram in enumerate(vocabulary)}

    # Vectorize documents
    X = np.zeros((len(docs), len(vocabulary)))
    for i, doc_ngrams in enumerate(processed_docs_ngrams):
        for ngram in doc_ngrams:
            if ngram in ngram_to_idx:
                X[i, ngram_to_idx[ngram]] += 1
    return X


# Training / Clustering

all_docs = [doc1, doc2, doc3, doc4, doc5, doc6, doc7, doc8]

# 1-gram Experiment
X1 = vectorize(all_docs, n_gram_size=1)
km1 = KMeans(n_clusters=2, random_state=42, n_init='auto').fit(X1)

# 2-gram Experiment
X2 = vectorize(all_docs, n_gram_size=2)
km2 = KMeans(n_clusters=2, random_state=42, n_init='auto').fit(X2)

print(f"1-gram clusters: {km1.labels_}")
print(f"2-gram clusters: {km2.labels_}")

y_true = [0, 0, 0, 0, 1, 1, 1, 1]
y_preds_one_gram  =  km1.labels_
y_preds_two_gram  =  km2.labels_


# compare accuracy and precision
print('Accuracy Kmeans(1-grams) : ',accuracy_score(y_true,y_preds_one_gram))
print('Accuracy Kmeans(2-grams) : ',accuracy_score(y_true,y_preds_two_gram))
print('Precision Kmeans(1-grams) : ',precision_score(y_true,y_preds_one_gram))
print('Precision Kmeans(2-grams) : ',precision_score(y_true,y_preds_two_gram))

1-gram clusters: [0 1 0 0 1 1 0 0]
2-gram clusters: [0 0 0 1 1 1 1 1]
Accuracy Kmeans(1-grams) :  0.625
Accuracy Kmeans(2-grams) :  0.875
Precision Kmeans(1-grams) :  0.6666666666666666
Precision Kmeans(2-grams) :  0.8
